In [ ]:
# Setup: import packages and define project paths.
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import os
import time
import matplotlib.colors as mcolors
from pyscenic.aucell import aucell
import gseapy as gp
from gseapy import SingleSampleGSEA
from collections import namedtuple
from scipy import sparse

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULT_DIR = PROJECT_DIR / "result"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)


In [ ]:
# Load the raw h5ad file.
file_path = DATA_DIR / "GSE201333_RAW" / "GSM6058681_TabulaSapiens.h5ad" / "GSM6058681_TabulaSapiens.h5ad"
adata = sc.read_h5ad(str(file_path))

print(adata.shape)
print(adata)


In [ ]:
# Plot UMAP colored by tissue and donor.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sc.pl.umap(
    adata,
    color='organ_tissue',
    title='UMAP by Organ Tissue',
    ax=axes[0],
    show=False
)

sc.pl.umap(
    adata,
    color='donor',
    title='UMAP by Donor',
    ax=axes[1],
    show=False
)

plt.tight_layout()
plt.show()


## Analysis overview

1. Use Scanpy to prepare files for BayesPrism deconvolution, including tissue labels, tissue-celltype labels, single-cell expression matrices, and cfRNA count matrices.

2. Use tissue- and cell-type-specific marker genes to improve the deconvolution reference matrix.

3. Convert the single-cell object into an R/Seurat-compatible format for downstream analysis.

4. Perform downstream deconvolution and module-score analysis in R.


In [ ]:
# Prepare the cfRNA count matrix for BayesPrism.
result = r.read_r(str(DATA_DIR / "WGCNA" / "no_combat" / "filtered_no_combat_counts.rds"))
counts_df = list(result.values())[0]

print("Counts matrix genes (first 5):", counts_df.index[:5].tolist())
print("Counts matrix samples (first 5):", counts_df.columns[:5].tolist())

# Match genes between cfRNA and single-cell data.
common_genes = counts_df.index.intersection(adata.var_names)
print(f"Number of shared genes: {len(common_genes)}")

adata = adata[:, common_genes].copy()
print("Filtered adata shape:", adata.shape)

counts_df = counts_df.loc[common_genes]
counts_deconv = counts_df.T
counts_deconv.to_csv(RESULT_DIR / "counts_no_combat_all_Deconvolution.csv")


In [ ]:
# Build BayesPrism single-cell reference files.
adata.obs['tissue_celltype'] = (
    adata.obs['organ_tissue'].astype(str)
    + '_'
    + adata.obs['cell_ontology_class'].astype(str)
)

counts = adata.obs['tissue_celltype'].value_counts()
print(counts.to_string())

# Filter low-abundance tissue-celltype groups.
keep_groups = counts[counts > 100].index
adata_filtered = adata[adata.obs['tissue_celltype'].isin(keep_groups)].copy()

print("Cells before filtering:", adata.n_obs)
print("Cells after filtering:", adata_filtered.n_obs)
print("Remaining tissue_celltype labels:", adata_filtered.obs['tissue_celltype'].unique())

# Export the raw-count reference matrix.
raw = adata_filtered.layers['raw_counts']
sparse.issparse(raw)
dense_counts = raw.toarray()

df_counts = pd.DataFrame(
    dense_counts,
    index=adata_filtered.obs_names,
    columns=adata_filtered.var_names
)

df_counts.to_csv(RESULT_DIR / "filtered_sc_counts_Deconvolution.csv")
df_counts.to_feather(RESULT_DIR / "filtered_sc_counts_Deconvolution.feather")

# Export tissue and tissue-celltype labels.
tissue_label = adata_filtered.obs['organ_tissue'].astype(str).values
tissue_celltype_label = adata_filtered.obs['tissue_celltype'].astype(str).values

np.savetxt(RESULT_DIR / "tissue_label.txt", tissue_label, fmt="%s")
np.savetxt(RESULT_DIR / "tissue_celltype_label.txt", tissue_celltype_label, fmt="%s")


In [ ]:
# Save the filtered single-cell object.
adata_filtered.write(RESULT_DIR / "adata_filtered.h5ad")
